# NLP SESSION 5

In [2]:
import os
import pandas as pd
import pickle
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import accuracy, NaiveBayesClassifier
from string import punctuation

MODEL_PATH = './model.pickle'
DATA_PATH = './financial_dataset.csv'

In [3]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [4]:
eng_stopwords = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [5]:
# PREPROCESSING
def getTag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'

def lemmatizing(tokens):
    lemmatized = []
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, getTag(tag))
        lemmatized.append(result)
    return lemmatized

def preprocessSentence(sentence):
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [word for word in tokens if word not in eng_stopwords]
    tokens = [word for word in tokens if word not in punctuation]
    tokens = [word for word in tokens if word.isalpha()]
    tokens = lemmatizing(tokens)
    return tokens

In [6]:
df = pd.read_csv(DATA_PATH)
x = df['Statement']
y = df['Sentiment']

allStatement = ' '.join(x)
allTokens = preprocessSentence(allStatement)
print(allTokens)

freqDist = FreqDist(allTokens)
print(freqDist)
freqDist

['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solution', 'provide', 'location', 'base', 'search', 'technology', 'community', 'platform', 'location', 'relevant', 'multimedia', 'content', 'new', 'powerful', 'commercial', 'model', 'esi', 'low', 'bk', 'real', 'possibility', 'last', 'quarter', 'componenta', 'net', 'sale', 'double', 'period', 'year', 'earlier', 'move', 'zero', 'profit', 'loss', 'spy', 'would', 'surprise', 'see', 'green', 'close', 'shell', 'billion', 'bg', 'deal', 'meet', 'shareholder', 'skepticism', 'ssh', 'communication', 'security', 'corp', 'stock', 'exchange', 'release', 'october', 'pm', 'company', 'update', 'full', 'year', 'outlook', 'estimate', 'result', 'remain', 'loss', 'full', 'year', 'kone', 'net', 'sale', 'rise', 'first', 'nine', 'month', 'circulation', 'revenue', 'increase', 'finland', 'sweden', 'sap', 'disappoints', 'software', 'license', 'real', 'problem', 'cloud', 'growth', 'trail', 'msft', 'orcl', 'goog', 'crm', 'adbe', 'http', 'subdivision', '

FreqDist({'eur': 747, 'mn': 459, 'profit': 368, 'sale': 338, 'company': 321, 'net': 297, 'say': 296, 'finnish': 266, 'million': 243, 'year': 234, ...})

In [7]:
def extract_feature(sentence):
    features = {}
    for word in freqDist.keys():
        features[word] = (word in sentence)
    return features

feature_sets = [(extract_feature(preprocessSentence(sentence)), sentiment)
                for sentence, sentiment in zip(x, y)
                ]
print(feature_sets[:1])

[({'geosolutions': True, 'technology': True, 'leverage': True, 'benefon': True, 'gps': True, 'solution': True, 'provide': True, 'location': True, 'base': True, 'search': True, 'community': True, 'platform': True, 'relevant': True, 'multimedia': True, 'content': True, 'new': True, 'powerful': True, 'commercial': True, 'model': True, 'esi': False, 'low': False, 'bk': False, 'real': False, 'possibility': False, 'last': False, 'quarter': False, 'componenta': False, 'net': False, 'sale': False, 'double': False, 'period': False, 'year': False, 'earlier': False, 'move': False, 'zero': False, 'profit': False, 'loss': False, 'spy': False, 'would': False, 'surprise': False, 'see': False, 'green': False, 'close': False, 'shell': False, 'billion': False, 'bg': False, 'deal': False, 'meet': False, 'shareholder': False, 'skepticism': False, 'ssh': False, 'communication': False, 'security': False, 'corp': False, 'stock': False, 'exchange': False, 'release': False, 'october': False, 'pm': False, 'comp

In [8]:
split = int(len(feature_sets) * 0.8)
train_set = feature_sets[:split]
test_set = feature_sets[split:]

In [9]:
# MENU CHOICE 1
def writeStatement():
    while True:
        statement = input("Please input your statement: ")
        if len(statement.split()) >= 2:
            return statement
        else:
            print("Invalid statement, should be at least contains 2 words")

def trainModel():
    # Training
    classifier = NaiveBayesClassifier.train(train_set)
    
    # Show 7 most Informative Features
    classifier.show_most_informative_features(7)
    
    # Count Accuracy
    acc = accuracy(classifier, test_set)
    print(f"Accuracy: {acc*100}%")
    
    # Save Model Pickle
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(classifier, f)
    return classifier
    

In [10]:
# MENU CHOICE 2

def show_pos_tag(statement):
    print('POS TAG')
    tokens = preprocessSentence(statement)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f'- {word}: {tag}')
    return tokens

def show_synonyms_antonyms(tokens):
    print()
    print("Synonyms and Antonyms")
    
    for token in tokens:
        synonyms = []
        antonyms = []
        
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                synonyms.append(lemma.name())
                
                for ant in lemma.antonyms():
                    antonyms.append(ant.name())
                    
        print(f"Word: {token}")
        if synonyms:
            print("Synonyms: ")
            print(f"{synonyms}")
        else:
            print("No synonyms")
            
        if antonyms:
            print("Antonyms: ")
            print(f"{antonyms}")
        else:
            print("No antonyms")

def predict_statement(classifier, tokens):
    feature = extract_feature(tokens)
    prediction = classifier.classify(feature)
    print(f"Prediction Statement: {prediction}")

def analyzeStatement(statement, classifier):
    if len(statement) == 0:
        print("Please input your statement on menu 1")
        return
    
    tokens = show_pos_tag(statement)
    show_synonyms_antonyms(tokens)
    predict_statement(classifier, tokens)
     

In [12]:
statement = ''

while True:
    print()
    print("1. Write your statement")
    print("2. Analyse your statement")
    print("3. Exit")
    choice = input(">> ")
    if choice == '1':
        statement = writeStatement()
        if os.path.exists(MODEL_PATH):
            with open(MODEL_PATH, 'rb') as f:
                classifier = pickle.load(f)
        else:
            classifier = trainModel()
                
    elif choice == '2':
        analyzeStatement(statement, classifier)
    
    elif choice == '3':
        break
    else:
        print("Invalid input")    


1. Write your statement
2. Analyse your statement
3. Exit
Most Informative Features
                decrease = True           negati : positi =     31.8 : 1.0
                    fell = True           negati : positi =     30.4 : 1.0
                     lay = True           negati : positi =     19.8 : 1.0
                   staff = True           negati : positi =     18.4 : 1.0
                  recall = True           negati : positi =     16.3 : 1.0
                   lower = True           negati : positi =     13.7 : 1.0
                    drop = True           negati : positi =     13.5 : 1.0
Accuracy: 77.90055248618785%

1. Write your statement
2. Analyse your statement
3. Exit
POS TAG
- budioni: NN
- siegar: NN

Synonyms and Antonyms
Word: budioni
No synonyms
No antonyms
Word: siegar
No synonyms
No antonyms
Prediction Statement: positive

1. Write your statement
2. Analyse your statement
3. Exit
